# Travel Reimbursement Approval Agent
**AI Developer Candidate Assignment | Runnable end-to-end prototype**

This notebook evaluates the five supplied sample claims against Appendix A. It uses a lightweight, tool-using agent architecture: the orchestrator calls policy lookup, receipt validation, per-diem calculation, duplicate detection, approval-tier evaluation, and output validation tools. Decisions are deterministic and traceable to policy IDs.

**Important design choice:** the policy engine—not an LLM—makes the final decision. This avoids hallucinated policy decisions. An LLM can optionally be added later to summarize evidence, but it must not override policy rules. No real employee or company data is used.

## 1. Setup

The core implementation uses Python's standard library and pandas/matplotlib for the results dashboard. No API key or paid service is required.

In [ ]:
import json
from datetime import date
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

## 2. Policy knowledge base (Appendix A)

The policy is stored as structured data so the agent can retrieve and cite stable policy IDs. Do not change these rules unless the assignment policy changes.

In [ ]:
POLICY = {
    "eligible_categories": {
        "airfare", "lodging", "meals", "ground_transport", "conference_fees"
    },
    "ineligible_categories": {
        "spa", "gym", "personal_entertainment", "minibar",
        "in_room_movies", "personal_shopping", "gifts",
        "traffic_fines", "penalties", "late_fees", "personal_expense"
    },
    "daily_limits": {
        "meals": 75.00,
        "lodging_per_night": 200.00,
        "ground_transport_per_day": 50.00
    },
    "receipt": {
        "threshold": 25.00,
        "airfare_always": True,
        "lodging_always": True
    },
    "approval_tiers": [
        {"id": "POL-APR-01", "name": "auto-approve", "min_exclusive": 0.00, "max_inclusive": 500.00},
        {"id": "POL-APR-02", "name": "manager", "min_exclusive": 500.00, "max_inclusive": 2000.00},
        {"id": "POL-APR-03", "name": "director-manual-review", "min_exclusive": 2000.00, "max_inclusive": float("inf")}
    ],
    "submission_window_days": 30,
    "policy_ids": {
        "eligible": "POL-CAT-01",
        "ineligible": "POL-CAT-02",
        "meals_limit": "POL-PD-01",
        "lodging_limit": "POL-PD-02",
        "transport_limit": "POL-PD-03",
        "airfare_class": "POL-AIR-01",
        "receipt_threshold": "POL-RCT-01",
        "missing_receipt": "POL-RCT-02",
        "submission_window": "POL-TIME-01",
    }
}

## 3. Sample claims (Appendix B)

All five claims below are transcribed from the supplied assignment. Amounts are USD. The trip and submission dates are included to evaluate the 30-day rule.

In [ ]:
CLAIMS = [
    {
        "claim_id": "CLM-001",
        "employee": "A. Rivera",
        "trip_start": "2026-06-10", "trip_end": "2026-06-12", "submitted": "2026-06-20",
        "items": [
            {"category":"airfare", "description":"Round-trip economy airfare", "amount":420.00, "receipt":True},
            {"category":"lodging", "description":"Hotel, 2 nights @ $180", "amount":360.00, "receipt":True, "nights":2},
            {"category":"meals", "description":"Meals, 3 days @ ~$60/day", "amount":180.00, "receipt":True, "days":3},
            {"category":"conference_fees", "description":"Conference registration", "amount":150.00, "receipt":True}
        ]
    },
    {
        "claim_id": "CLM-002",
        "employee": "B. Osei",
        "trip_start": "2026-06-14", "trip_end": "2026-06-15", "submitted": "2026-06-25",
        "items": [
            {"category":"spa", "description":"Hotel spa package", "amount":300.00, "receipt":True},
            {"category":"minibar", "description":"In-room minibar", "amount":80.00, "receipt":True}
        ]
    },
    {
        "claim_id": "CLM-003",
        "employee": "C. Nakamura",
        "trip_start": "2026-06-08", "trip_end": "2026-06-10", "submitted": "2026-06-22",
        "items": [
            {"category":"airfare", "description":"Round-trip economy airfare", "amount":300.00, "receipt":True},
            {"category":"lodging", "description":"Hotel, 2 nights @ $250", "amount":500.00, "receipt":True, "nights":2},
            {"category":"meals", "description":"Meals, 2 days @ $70/day", "amount":140.00, "receipt":True, "days":2}
        ]
    },
    {
        "claim_id": "CLM-004",
        "employee": "D. Fischer",
        "trip_start": "2026-06-16", "trip_end": "2026-06-18", "submitted": "2026-06-28",
        "items": [
            {"category":"airfare", "description":"Business-class international airfare", "amount":2400.00, "receipt":True, "airfare_class":"business"},
            {"category":"lodging", "description":"Hotel, 3 nights", "amount":600.00, "receipt":False, "nights":3}
        ]
    },
    {
        "claim_id": "CLM-005",
        "employee": "E. Haddad",
        "trip_start": "2026-06-11", "trip_end": "2026-06-11", "submitted": "2026-06-24",
        "items": [
            {"category":"meals", "description":"Client dinner for 4 (business development)", "amount":220.00, "receipt":False, "days":1}
        ]
    }
]

# Quick input integrity check
assert len(CLAIMS) == 5
for claim in CLAIMS:
    assert all(item["amount"] >= 0 for item in claim["items"])

## 4. Agent tools

Each function is a small, independently testable tool. The agent records which tools it used for every claim.

- `policy_lookup`: retrieves the applicable policy rule and ID.
- `receipt_completeness_check`: identifies missing mandatory receipts.
- `per_diem_limit_checker`: computes allowable amount and excess per item.
- `duplicate_detector`: flags repeated identical line items (the input does not provide vendor/transaction IDs, so this is only a conservative same-category/amount/description check).
- `approval_threshold_checker`: applies the approval bands to the post-cap eligible amount.
- `output_validator`: enforces the required JSON schema and decision vocabulary.

In [ ]:
def policy_lookup(category):
    if category in POLICY["ineligible_categories"]:
        return {"eligible": False, "policy_id": POLICY["policy_ids"]["ineligible"], "reason": "Ineligible expense category"}
    if category in POLICY["eligible_categories"]:
        return {"eligible": True, "policy_id": POLICY["policy_ids"]["eligible"], "reason": "Eligible expense category"}
    return {"eligible": None, "policy_id": None, "reason": "Unknown category; needs manual review"}


def receipt_completeness_check(item):
    amount = float(item["amount"])
    category = item["category"]
    always_required = category in ("airfare", "lodging")
    required_by_threshold = amount > POLICY["receipt"]["threshold"]
    required = always_required or required_by_threshold
    missing = required and not bool(item.get("receipt", False))
    return {
        "required": required,
        "missing": missing,
        "policy_id": POLICY["policy_ids"]["missing_receipt"] if missing else (
            POLICY["policy_ids"]["receipt_threshold"] if required else None
        )
    }


def per_diem_limit_checker(item):
    category, amount = item["category"], float(item["amount"])
    if category == "meals":
        cap = POLICY["daily_limits"]["meals"] * int(item.get("days", 1))
        policy_id = POLICY["policy_ids"]["meals_limit"]
    elif category == "lodging":
        cap = POLICY["daily_limits"]["lodging_per_night"] * int(item.get("nights", 1))
        policy_id = POLICY["policy_ids"]["lodging_limit"]
    elif category == "ground_transport":
        cap = POLICY["daily_limits"]["ground_transport_per_day"] * int(item.get("days", 1))
        policy_id = POLICY["policy_ids"]["transport_limit"]
    else:
        return {"cap": amount, "allowable": amount, "excess": 0.0, "policy_id": None}
    allowable = min(amount, cap)
    return {
        "cap": round(cap, 2),
        "allowable": round(allowable, 2),
        "excess": round(max(0.0, amount - cap), 2),
        "policy_id": policy_id
    }


def duplicate_detector(items):
    seen, duplicates = set(), []
    for index, item in enumerate(items):
        key = (item["category"], round(float(item["amount"]), 2), item["description"].strip().lower())
        if key in seen:
            duplicates.append(index)
        seen.add(key)
    return duplicates


def approval_threshold_checker(amount):
    amount = float(amount)
    for tier in POLICY["approval_tiers"]:
        if amount > tier["min_exclusive"] and amount <= tier["max_inclusive"]:
            return tier
    # A zero-dollar claim does not qualify for an approval tier.
    return {"id": None, "name": "zero_or_invalid", "min_exclusive": 0, "max_inclusive": 0}


def output_validator(result):
    required = {
        "claim_id", "decision", "original_amount", "approved_amount",
        "deducted_amount", "missing_docs", "policy_refs", "confidence",
        "explanation", "tools_used"
    }
    assert set(result.keys()) == required, f"Output keys mismatch: {set(result.keys()) ^ required}"
    assert result["decision"] in {"APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"}
    for key in ("original_amount", "approved_amount", "deducted_amount", "confidence"):
        assert isinstance(result[key], (int, float))
    assert 0 <= result["confidence"] <= 1
    assert isinstance(result["missing_docs"], list)
    assert isinstance(result["policy_refs"], list)
    assert isinstance(result["tools_used"], list)
    return True

## 5. Agent orchestration and decision policy

Decision precedence is intentionally conservative:

1. Unknown categories, policy exceptions (such as business-class airfare), missing required receipts, duplicate suspicion, late submission, or a claim above $2,000 trigger `MANUAL_REVIEW`.
2. If every item is ineligible, return `REJECT`.
3. If any eligible item exceeds a per-diem cap, return `PARTIAL_APPROVE` and deduct only the excess.
4. Otherwise return `APPROVE`.

The approval tier is checked after per-diem deductions. A claim in the director tier is routed to manual review even if its line items otherwise comply. For a manual-review claim, `approved_amount` is **zero until a reviewer approves it**; the explanation may state the calculated amount pending review. This avoids representing a provisional calculation as an approval.

In [ ]:
class TravelReimbursementAgent:
    def evaluate(self, claim):
        tools_used = []
        refs = set()
        missing_docs = []
        manual_reasons = []
        line_results = []

        # Tool 1: duplicate check
        tools_used.append("duplicate_detector")
        duplicate_indices = duplicate_detector(claim["items"])
        if duplicate_indices:
            manual_reasons.append(f"Possible duplicate line item(s) at index {duplicate_indices}")

        # Submission-window check
        tools_used.append("submission_window_check")
        days_after_trip = (date.fromisoformat(claim["submitted"]) - date.fromisoformat(claim["trip_end"])).days
        if days_after_trip > POLICY["submission_window_days"]:
            refs.add(POLICY["policy_ids"]["submission_window"])
            manual_reasons.append(f"Submitted {days_after_trip} days after trip end; exceeds 30 days")

        # Evaluate each line item
        for item in claim["items"]:
            # Tool 2: policy lookup
            tools_used.append("policy_lookup")
            policy = policy_lookup(item["category"])
            if policy["policy_id"]:
                refs.add(policy["policy_id"])

            if policy["eligible"] is None:
                manual_reasons.append(f"Unknown expense category: {item['category']}")
                line_results.append({"eligible": False, "allowable": 0.0, "excess": 0.0})
                continue

            if not policy["eligible"]:
                line_results.append({"eligible": False, "allowable": 0.0, "excess": float(item["amount"])})
                continue

            # Business/first-class airfare is a policy exception, not an automatic deduction.
            if item["category"] == "airfare" and item.get("airfare_class", "economy").lower() != "economy":
                refs.add(POLICY["policy_ids"]["airfare_class"])
                manual_reasons.append("Business/first-class airfare is a policy exception; pre-approval must be checked")

            # Tool 3: receipt completeness
            tools_used.append("receipt_completeness_check")
            receipt = receipt_completeness_check(item)
            if receipt["policy_id"]:
                refs.add(receipt["policy_id"])
            if receipt["missing"]:
                missing_docs.append(f"Receipt for {item['category']}: {item['description']}")
                manual_reasons.append(f"Required receipt missing for {item['category']}")

            # Tool 4: per-diem checker
            tools_used.append("per_diem_limit_checker")
            limit = per_diem_limit_checker(item)
            if limit["policy_id"]:
                refs.add(limit["policy_id"])
            line_results.append({
                "eligible": True,
                "allowable": limit["allowable"],
                "excess": limit["excess"]
            })

        original = round(sum(float(i["amount"]) for i in claim["items"]), 2)
        eligible_items = [r for r in line_results if r["eligible"]]
        eligible_count = len(eligible_items)
        all_ineligible = eligible_count == 0 and not manual_reasons

        # Calculate the amount that would be reimbursable if no manual-review blocker existed.
        calculated_allowable = round(sum(r["allowable"] for r in eligible_items), 2)
        cap_excess = round(sum(r["excess"] for r in eligible_items), 2)

        # Tool 5: approval threshold check
        tools_used.append("approval_threshold_checker")
        tier = approval_threshold_checker(calculated_allowable)
        if tier["id"]:
            refs.add(tier["id"])
        if tier["id"] == "POL-APR-03":
            manual_reasons.append("Post-cap amount exceeds $2,000 director/manual-review threshold")

        # Decide
        if all_ineligible:
            decision = "REJECT"
            approved = 0.0
            deducted = original
            explanation = "All claimed expenses are ineligible under policy. No reimbursement is allowed."
        elif manual_reasons:
            decision = "MANUAL_REVIEW"
            approved = 0.0
            # Report only known policy cap deductions; unresolved items are not treated as finally approved.
            deducted = cap_excess
            explanation = (
                "Manual review is required: " + "; ".join(dict.fromkeys(manual_reasons)) +
                f". The calculated eligible amount before unresolved review is ${calculated_allowable:.2f}; "
                "approved_amount remains $0.00 until a reviewer makes the final decision."
            )
        elif eligible_count == 0:
            decision = "REJECT"
            approved = 0.0
            deducted = original
            explanation = "No eligible expenses remain."
        elif cap_excess > 0:
            decision = "PARTIAL_APPROVE"
            approved = calculated_allowable
            deducted = round(original - approved, 2)
            explanation = (
                f"Eligible expenses exceed one or more per-diem limits. Reimburse ${approved:.2f} "
                f"and deduct ${deducted:.2f}. Post-cap amount falls in the {tier['name']} approval tier."
            )
        else:
            decision = "APPROVE"
            approved = calculated_allowable
            deducted = round(original - approved, 2)
            explanation = (
                f"All items are eligible, required receipts are present, limits are met, and the "
                f"${approved:.2f} post-cap amount is within the {tier['name']} approval tier."
            )

        # Confidence reflects whether the policy inputs are complete and unambiguous.
        confidence = 0.98 if decision in ("APPROVE", "REJECT") else (
            0.94 if decision == "PARTIAL_APPROVE" else 0.78
        )
        result = {
            "claim_id": claim["claim_id"],
            "decision": decision,
            "original_amount": original,
            "approved_amount": round(approved, 2),
            "deducted_amount": round(deducted, 2),
            "missing_docs": missing_docs,
            "policy_refs": sorted(refs),
            "confidence": confidence,
            "explanation": explanation,
            "tools_used": sorted(set(tools_used + ["output_validator"]))
        }
        # Tool 6: output validation
        output_validator(result)
        return result


agent = TravelReimbursementAgent()
RESULTS = [agent.evaluate(claim) for claim in CLAIMS]
print(json.dumps(RESULTS, indent=2))

## 6. Required structured output

The next cell creates exactly one JSON result object per supplied claim. It can be copied directly into a downstream API or saved as a file.

In [ ]:
results_json = json.dumps(RESULTS, indent=2)
with open("travel_reimbursement_results.json", "w", encoding="utf-8") as f:
    f.write(results_json)

print(results_json)

## 7. Results dashboard

The dashboard is generated from the actual agent outcomes, not manually entered summary numbers.

In [ ]:
summary = pd.DataFrame(RESULTS)
display(summary[["claim_id", "decision", "original_amount", "approved_amount", "deducted_amount", "confidence"]])

decision_counts = summary["decision"].value_counts().reindex(
    ["APPROVE", "PARTIAL_APPROVE", "REJECT", "MANUAL_REVIEW"], fill_value=0
)
display(decision_counts.rename("claim_count").to_frame())

decision_counts.plot(kind="bar", title="Travel claim decision breakdown", xlabel="Decision", ylabel="Number of claims", rot=20)
plt.tight_layout()
plt.show()

amounts = summary.set_index("claim_id")[["original_amount", "approved_amount", "deducted_amount"]]
amounts.plot(kind="bar", title="Claimed vs approved vs deducted amounts (USD)", ylabel="USD", rot=0)
plt.tight_layout()
plt.show()

## 8. Line-item audit trail

This view helps a reviewer understand how each expense was treated. It is useful during the interview because every deduction or escalation can be tied back to a specific rule.

In [ ]:
audit_rows = []
for claim in CLAIMS:
    for item in claim["items"]:
        p = policy_lookup(item["category"])
        r = receipt_completeness_check(item)
        limit = per_diem_limit_checker(item) if p["eligible"] else {"allowable":0.0, "excess":float(item["amount"]), "cap":0.0}
        audit_rows.append({
            "claim_id": claim["claim_id"],
            "category": item["category"],
            "description": item["description"],
            "claimed_usd": round(float(item["amount"]), 2),
            "eligible": p["eligible"],
            "cap_usd": limit.get("cap"),
            "calculated_allowable_usd": limit["allowable"],
            "excess_usd": limit["excess"],
            "receipt_required": r["required"],
            "receipt_missing": r["missing"],
            "policy_id": p["policy_id"]
        })
audit_df = pd.DataFrame(audit_rows)
display(audit_df)

## 9. Basic tests

These checks protect against common regressions: decision vocabulary, exact result count, arithmetic consistency for completed approvals, and expected high-level outcomes for the five supplied examples.

In [ ]:
assert len(RESULTS) == 5
assert [r["decision"] for r in RESULTS] == [
    "APPROVE", "REJECT", "PARTIAL_APPROVE", "MANUAL_REVIEW", "MANUAL_REVIEW"
]
assert RESULTS[0]["approved_amount"] == 1110.00
assert RESULTS[1]["approved_amount"] == 0.00
assert RESULTS[2]["approved_amount"] == 840.00
assert RESULTS[2]["deducted_amount"] == 100.00
assert RESULTS[3]["approved_amount"] == 0.00  # pending manual review
assert RESULTS[4]["approved_amount"] == 0.00  # pending missing receipt review
for result in RESULTS:
    output_validator(result)
print("All tests passed.")

## 10. Design Notes & Reasoning

### Assumptions
- The trip end date is used as the expense-date reference for the 30-day submission-window check because individual expense dates are not supplied.
- A missing required receipt routes the claim to Manual Review; it is not silently rejected.
- Meals and ground transport use the provided `days` count; lodging uses `nights`. The sample claims include these counts in their descriptions.
- Business-class airfare is not automatically deducted. It is a policy exception and must be reviewed for pre-approval.
- `approved_amount` means finally approved by this prototype. For Manual Review cases it is zero until the reviewer decides; the explanation contains the computed eligible amount where useful.
- The client dinner in CLM-005 is treated as a meals expense because the provided policy does not define a separate client-entertainment rule. The daily meal cap still applies.

### Trade-offs
- The prototype uses deterministic Python functions rather than requiring a paid LLM/API. This makes it runnable, repeatable, testable, and auditable in a free-tier environment.
- The agent is tool-oriented: it retrieves policy, checks receipts, applies caps, checks approval tiers, and validates output. A language model can be added for natural-language intake or explanation, but should never be allowed to invent or override policy.
- Duplicate detection is deliberately conservative. The sample data has no vendor, receipt ID, or transaction ID, so the detector can only identify exact repeated line-item descriptions/categories/amounts. Production duplicate detection would need stable transaction metadata.
- The dashboard is intentionally small and data-driven to meet the assignment's lightweight prototype requirement.

### Known gaps
- No OCR is implemented for receipt images; receipt presence is represented by the supplied Yes/No field.
- The policy does not specify how to prorate partial travel days, how to treat shared lodging, or how to validate business purpose beyond the supplied descriptions.
- The policy says approval is eligible at the manager tier when fully compliant; this prototype does not simulate an actual manager approval workflow.
- The source does not specify a separate rule for client meals, tips, taxes, currency conversion, or expense dates per line item.

### What I would improve next
1. Add receipt upload and OCR extraction with human confirmation.
2. Add a persistent audit log with policy version, tool inputs, timestamps, and reviewer actions.
3. Add vendor/receipt IDs for reliable duplicate detection.
4. Add a small Streamlit UI or API endpoint after validating the notebook workflow.
5. Add an optional LLM explanation layer with strict JSON schema and policy-grounded retrieval; keep the deterministic policy engine as the final authority.

### How to run
1. Open this notebook in Jupyter Notebook, JupyterLab, or Google Colab.
2. Run all cells from top to bottom.
3. Review the structured JSON output, dashboard, audit table, and passing tests.
4. The notebook writes `travel_reimbursement_results.json` in the current working directory.

## 11. Optional GenAI tool-calling layer (requires an API key)

The notebook above is fully runnable without an LLM. To demonstrate a real GenAI tool-calling workflow, this optional cell lets an LLM decide to call the deterministic `evaluate_claim` tool for a selected claim. The tool returns the policy engine's structured result. The LLM may summarize it, but it is explicitly prohibited from changing the decision or amounts.

To enable it, install `openai`, set `OPENAI_API_KEY` in your environment, and run this cell. API usage may incur charges. Set `OPENAI_MODEL` to a model available to your API account. This follows the Responses API function-calling pattern.

In [ ]:
# Optional: real LLM tool calling. Skips safely when no API key is configured.
import os

if not os.getenv('OPENAI_API_KEY'):
    print('Skipped: set OPENAI_API_KEY to enable the optional GenAI layer. The deterministic agent and tests above are complete.')
else:
    # If needed, install once: %pip install -q openai
    from openai import OpenAI
    client = OpenAI()
    selected_claim_id = 'CLM-004'
    tool_schema = [{
        'type': 'function',
        'name': 'evaluate_claim',
        'description': 'Run the deterministic travel reimbursement policy engine for one supplied claim. Never modify its decision or amounts.',
        'parameters': {
            'type': 'object',
            'properties': {'claim_id': {'type': 'string', 'enum': [c['claim_id'] for c in CLAIMS]}},
            'required': ['claim_id'],
            'additionalProperties': False
        },
        'strict': True
    }]
    user_prompt = f'Evaluate claim {selected_claim_id}. Use the evaluate_claim tool. Explain the tool result clearly. Do not change the returned decision, amounts, or policy references.'
    first = client.responses.create(
        model=os.getenv('OPENAI_MODEL', 'gpt-5.5'),
        input=user_prompt,
        tools=tool_schema,
        tool_choice='auto'
    )
    tool_outputs = []
    for call in first.output:
        if getattr(call, 'type', None) == 'function_call' and call.name == 'evaluate_claim':
            args = json.loads(call.arguments)
            chosen = next(c for c in CLAIMS if c['claim_id'] == args['claim_id'])
            deterministic_result = agent.evaluate(chosen)
            tool_outputs.append({
                'type': 'function_call_output',
                'call_id': call.call_id,
                'output': json.dumps(deterministic_result)
            })
    if not tool_outputs:
        print('The model did not request the evaluation tool. First response:', first.output_text)
    else:
        second = client.responses.create(
            model=os.getenv('OPENAI_MODEL', 'gpt-5.5'),
            input=[{'role': 'user', 'content': user_prompt}, *first.output, *tool_outputs],
            tools=tool_schema
        )
        print(second.output_text)
        print('\nAuthoritative structured result from the policy engine:')
        print(json.dumps(deterministic_result, indent=2))

### GenAI safety boundary

- The LLM can choose when to request a claim evaluation and can explain the returned evidence.
- The Python policy engine owns eligibility, limits, receipts, approval thresholds, and final status.
- If the LLM response conflicts with the tool output, the structured tool output wins.
- For production, add retries, API timeout handling, model/version logging, PII controls, and human approval before any payment.